In [1]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
    print("GPU memory (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
    print("\nGPU confirmed. Safe to continue.")
else:
    print("\nWARNING: No GPU detected.")
    print("Go to Runtime > Change runtime type > T4 GPU, then Factory reset runtime and rerun this cell.")

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU name: Tesla T4
GPU memory (GB): 15.6

GPU confirmed. Safe to continue.


In [2]:
from google.colab import drive
drive.mount("/content/drive")

import os

expected_files = [
    "/content/drive/MyDrive/fraud-detector-fyp/data/processed/train.csv",
    "/content/drive/MyDrive/fraud-detector-fyp/data/processed/val.csv",
    "/content/drive/MyDrive/fraud-detector-fyp/data/processed/test.csv",
]

print("Checking uploaded files:")
all_found = True
for path in expected_files:
    exists = os.path.exists(path)
    status = "FOUND" if exists else "MISSING"
    print(f"  {status}: {path}")
    if not exists:
        all_found = False

if all_found:
    print("\nAll files confirmed. Safe to continue.")
else:
    print("\nWARNING: One or more files missing. Check your Drive folder structure before continuing.")

Mounted at /content/drive
Checking uploaded files:
  FOUND: /content/drive/MyDrive/fraud-detector-fyp/data/processed/train.csv
  FOUND: /content/drive/MyDrive/fraud-detector-fyp/data/processed/val.csv
  FOUND: /content/drive/MyDrive/fraud-detector-fyp/data/processed/test.csv

All files confirmed. Safe to continue.


In [3]:
import subprocess
import sys

packages = [
    "transformers==4.40.0",
    "accelerate>=0.26.0",
]

print("Installing required packages...")
for package in packages:
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", package, "-q"],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print(f"  OK: {package}")
    else:
        print(f"  FAILED: {package}")
        print(result.stderr)

print("\nVerifying imports:")
import transformers
import accelerate
print(f"  transformers version: {transformers.__version__}")
print(f"  accelerate version: {accelerate.__version__}")
print("\nAll packages ready.")

Installing required packages...
  OK: transformers==4.40.0
  OK: accelerate>=0.26.0

Verifying imports:
  transformers version: 4.40.0
  accelerate version: 1.14.0

All packages ready.


In [4]:
# Do NOT upgrade transformers — work with Colab's native version.
# Only upgrade accelerate, which has no system-level conflicts.
!pip install --upgrade accelerate -q

# Uninstall peft entirely — it's pre-installed in Colab but we don't
# need it for BERT fine-tuning, and it's the source of the import conflict.
!pip uninstall peft -y -q

import importlib
import sys

# Clear any cached transformers/peft modules from this session
mods_to_remove = [k for k in sys.modules if 'transformers' in k or 'peft' in k]
for mod in mods_to_remove:
    del sys.modules[mod]

import transformers
import accelerate
print(f"transformers version: {transformers.__version__}")
print(f"accelerate version: {accelerate.__version__}")

# Verify Trainer imports cleanly without peft interference
from transformers import Trainer
print("Trainer import: OK")
print("Safe to continue.")

transformers version: 4.40.0
accelerate version: 1.14.0
Trainer import: OK
Safe to continue.


In [5]:
import os
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from transformers import (
    BertTokenizerFast,
    BertForSequenceClassification,
    Trainer,
    TrainingArguments,
)
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("All imports successful")
print("Training device:", device)

BASE_DIR = "/content/drive/MyDrive/fraud-detector-fyp"
DATA_DIR = os.path.join(BASE_DIR, "data", "processed")
MODEL_DIR = os.path.join(BASE_DIR, "saved_models", "bert_fraud_detector")

os.makedirs(MODEL_DIR, exist_ok=True)

print("Data directory:", DATA_DIR)
print("Model save directory:", MODEL_DIR)

All imports successful
Training device: cuda
Data directory: /content/drive/MyDrive/fraud-detector-fyp/data/processed
Model save directory: /content/drive/MyDrive/fraud-detector-fyp/saved_models/bert_fraud_detector


In [6]:
train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"), index_col="original_index")
val_df   = pd.read_csv(os.path.join(DATA_DIR, "val.csv"),   index_col="original_index")
test_df  = pd.read_csv(os.path.join(DATA_DIR, "test.csv"),  index_col="original_index")

print("Shapes:")
print("  Train:", train_df.shape)
print("  Val:  ", val_df.shape)
print("  Test: ", test_df.shape)

print("\nClass balance:")
for name, d in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    fraud_count = d["fraudulent"].sum()
    fraud_pct   = d["fraudulent"].mean() * 100
    print(f"  {name}: {fraud_count} fraudulent / {len(d)} total ({fraud_pct:.2f}%)")

print("\nNulls in clean_text:")
for name, d in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    print(f"  {name}: {d['clean_text'].isnull().sum()}")

print("\nPlaceholder residue (should be 0):")
pattern = r"#[a-z]+_[0-9a-f]+#"
for name, d in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    residue = d["clean_text"].str.contains(pattern, regex=True).sum()
    print(f"  {name}: {residue}")

print("\nfraudulent dtype:", train_df["fraudulent"].dtype)
print("\nData verified. Safe to continue.")

Shapes:
  Train: (12516, 3)
  Val:   (2682, 3)
  Test:  (2682, 3)

Class balance:
  Train: 606 fraudulent / 12516 total (4.84%)
  Val: 130 fraudulent / 2682 total (4.85%)
  Test: 130 fraudulent / 2682 total (4.85%)

Nulls in clean_text:
  Train: 0
  Val: 0
  Test: 0

Placeholder residue (should be 0):
  Train: 0
  Val: 0
  Test: 0

fraudulent dtype: int64

Data verified. Safe to continue.


In [7]:
tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")

def tokenize_texts(texts):
    return tokenizer(
        texts,
        truncation=True,
        padding=True,
        max_length=512,
        return_tensors="pt",
    )

print("Tokenizing training set...")
train_encodings = tokenize_texts(train_df["clean_text"].tolist())
print("Tokenizing validation set...")
val_encodings   = tokenize_texts(val_df["clean_text"].tolist())
print("Tokenizing test set...")
test_encodings  = tokenize_texts(test_df["clean_text"].tolist())

print("\nTokenization complete.")
print("Train input_ids shape:", train_encodings["input_ids"].shape)
print("Val input_ids shape:  ", val_encodings["input_ids"].shape)
print("Test input_ids shape: ", test_encodings["input_ids"].shape)

# Verify padding is consistent - all sequences should be max_length
# (padding=True pads to the longest sequence in the batch,
# which here is the entire dataset, so most will pad to 512)
max_len_train = train_encodings["input_ids"].shape[1]
print(f"\nActual sequence length used: {max_len_train} tokens")
if max_len_train == 512:
    print("All sequences padded/truncated to max_length=512")
else:
    print(f"Longest sequence in dataset is {max_len_train} tokens — padding to this length")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

Tokenizing training set...
Tokenizing validation set...
Tokenizing test set...

Tokenization complete.
Train input_ids shape: torch.Size([12516, 512])
Val input_ids shape:   torch.Size([2682, 512])
Test input_ids shape:  torch.Size([2682, 512])

Actual sequence length used: 512 tokens
All sequences padded/truncated to max_length=512


In [8]:
class JobPostingDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels    = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {
            key: val[idx].clone().detach()
            for key, val in self.encodings.items()
        }
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

train_labels = train_df["fraudulent"].tolist()
val_labels   = val_df["fraudulent"].tolist()
test_labels  = test_df["fraudulent"].tolist()

train_dataset = JobPostingDataset(train_encodings, train_labels)
val_dataset   = JobPostingDataset(val_encodings,   val_labels)
test_dataset  = JobPostingDataset(test_encodings,  test_labels)

print("Dataset sizes:")
print("  Train:", len(train_dataset))
print("  Val:  ", len(val_dataset))
print("  Test: ", len(test_dataset))

sample = train_dataset[0]
print("\nSample item keys:", list(sample.keys()))
print("input_ids shape:", sample["input_ids"].shape)
print("attention_mask shape:", sample["attention_mask"].shape)
print("label:", sample["labels"].item())

Dataset sizes:
  Train: 12516
  Val:   2682
  Test:  2682

Sample item keys: ['input_ids', 'token_type_ids', 'attention_mask', 'labels']
input_ids shape: torch.Size([512])
attention_mask shape: torch.Size([512])
label: 0


In [9]:
CLASS_WEIGHT_FRAUD = 19.65

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2,
)
model.to(device)
print("Model loaded and moved to:", device)
print("Trainable parameters:", f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        class_weights = torch.tensor(
            [1.0, CLASS_WEIGHT_FRAUD],
            dtype=torch.float,
            device=logits.device,
        )
        loss_fn = nn.CrossEntropyLoss(weight=class_weights)
        loss = loss_fn(logits, labels)

        return (loss, outputs) if return_outputs else loss

print("WeightedTrainer defined successfully")
print(f"Class weights: genuine=1.0, fraudulent={CLASS_WEIGHT_FRAUD}")

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded and moved to: cuda
Trainable parameters: 109,483,778
WeightedTrainer defined successfully
Class weights: genuine=1.0, fraudulent=19.65


In [10]:
import numpy as np
from sklearn.metrics import recall_score, precision_score, f1_score, accuracy_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    return {
        "accuracy":  accuracy_score(labels, predictions),
        "precision": precision_score(labels, predictions, zero_division=0),
        "recall":    recall_score(labels, predictions, zero_division=0),
        "f1":        f1_score(labels, predictions, zero_division=0),
    }

CHECKPOINT_DIR = os.path.join(MODEL_DIR, "checkpoints")

training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="recall",
    greater_is_better=True,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    logging_steps=50,
    fp16=True,
    dataloader_num_workers=2,
    report_to="none",
)

print("TrainingArguments configured successfully")
print(f"Epochs: {training_args.num_train_epochs}")
print(f"Train batch size: {training_args.per_device_train_batch_size}")
print(f"Eval batch size: {training_args.per_device_eval_batch_size}")
print(f"Best model metric: {training_args.metric_for_best_model}")
print(f"fp16 (mixed precision): {training_args.fp16}")
print(f"Checkpoint directory: {CHECKPOINT_DIR}")

TrainingArguments configured successfully
Epochs: 4
Train batch size: 16
Eval batch size: 32
Best model metric: recall
fp16 (mixed precision): True
Checkpoint directory: /content/drive/MyDrive/fraud-detector-fyp/saved_models/bert_fraud_detector/checkpoints


In [11]:
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

print("Starting BERT fine-tuning...")
print(f"Training on {len(train_dataset)} examples for {training_args.num_train_epochs} epochs")
print(f"Evaluating on {len(val_dataset)} examples after each epoch")
print(f"Best model (by validation recall) will be saved to: {MODEL_DIR}")
print("="*60)

trainer.train()

print("="*60)
print("Training complete.")
print(f"Best model checkpoint saved to: {training_args.output_dir}")

Starting BERT fine-tuning...
Training on 12516 examples for 4 epochs
Evaluating on 2682 examples after each epoch
Best model (by validation recall) will be saved to: /content/drive/MyDrive/fraud-detector-fyp/saved_models/bert_fraud_detector


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.425700,0.415874,0.980239,0.873786,0.692308,0.772532


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.425700,0.415874,0.980239,0.873786,0.692308,0.772532
2,0.198400,0.389001,0.988441,0.938053,0.815385,0.872428
3,0.047600,0.413352,0.987323,0.936364,0.792308,0.858333
4,0.222800,0.365357,0.986577,0.879032,0.838462,0.858268


Training complete.
Best model checkpoint saved to: /content/drive/MyDrive/fraud-detector-fyp/saved_models/bert_fraud_detector/checkpoints


In [12]:
print("Saving best model and tokenizer to Drive...")

trainer.save_model(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)

metadata = {
    "model_type": "BertForSequenceClassification",
    "base_model": "bert-base-uncased",
    "num_labels": 2,
    "label_map": {"0": "genuine", "1": "fraudulent"},
    "max_length": 512,
    "decision_threshold": 0.5,
    "class_weight_fraudulent": 19.65,
    "training_config": {
        "epochs": 4,
        "batch_size": 16,
        "learning_rate": 2e-5,
        "weight_decay": 0.01,
        "warmup_ratio": 0.1,
        "fp16": True,
        "best_epoch_by_recall": 4,
    },
    "validation_metrics_at_default_threshold": {
        "accuracy": 0.9866,
        "precision_fraud": 0.879,
        "recall_fraud": 0.838,
        "f1_fraud": 0.858,
    },
    "comparison_vs_baseline": {
        "best_baseline": "Random Forest @ threshold 0.3",
        "baseline_recall": 0.831,
        "baseline_precision": 0.812,
        "baseline_f1": 0.821,
        "bert_recall": 0.838,
        "bert_precision": 0.879,
        "bert_f1": 0.858,
    },
    "training_environment": "Google Colab T4 GPU",
    "training_duration_minutes": 33,
}

metadata_path = os.path.join(MODEL_DIR, "bert_metadata.json")
with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2)

print("Saved:")
print(f"  Model weights: {MODEL_DIR}/model.safetensors")
print(f"  Tokenizer:     {MODEL_DIR}/tokenizer files")
print(f"  Metadata:      {metadata_path}")

saved_files = os.listdir(MODEL_DIR)
print(f"\nFiles in {MODEL_DIR}:")
for fname in sorted(saved_files):
    if not os.path.isdir(os.path.join(MODEL_DIR, fname)):
        size_mb = os.path.getsize(os.path.join(MODEL_DIR, fname)) / 1e6
        print(f"  {fname} ({size_mb:.1f} MB)")

Saving best model and tokenizer to Drive...
Saved:
  Model weights: /content/drive/MyDrive/fraud-detector-fyp/saved_models/bert_fraud_detector/model.safetensors
  Tokenizer:     /content/drive/MyDrive/fraud-detector-fyp/saved_models/bert_fraud_detector/tokenizer files
  Metadata:      /content/drive/MyDrive/fraud-detector-fyp/saved_models/bert_fraud_detector/bert_metadata.json

Files in /content/drive/MyDrive/fraud-detector-fyp/saved_models/bert_fraud_detector:
  bert_metadata.json (0.0 MB)
  config.json (0.0 MB)
  model.safetensors (438.0 MB)
  special_tokens_map.json (0.0 MB)
  tokenizer.json (0.7 MB)
  tokenizer_config.json (0.0 MB)
  training_args.bin (0.0 MB)
  vocab.txt (0.2 MB)


In [13]:
print("Running threshold tuning on validation set...")

val_outputs = trainer.predict(val_dataset)
val_logits = val_outputs.predictions
val_probs = torch.softmax(torch.tensor(val_logits), dim=1).numpy()
val_probs_fraud = val_probs[:, 1]

thresholds = [0.5, 0.4, 0.3, 0.2, 0.1]
threshold_results = []

print(f"\n{'Threshold':<12} {'Precision':>10} {'Recall':>10} {'F1':>10} {'FN count':>10} {'FP count':>10}")
print("-" * 62)

for t in thresholds:
    preds = (val_probs_fraud >= t).astype(int)

    tp = ((preds == 1) & (val_labels == 1)).sum()
    fp = ((preds == 1) & (val_labels == 0)).sum()
    fn = ((preds == 0) & (val_labels == 1)).sum()

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    threshold_results.append({
        "threshold": t,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "fn_count": int(fn),
        "fp_count": int(fp),
    })

    print(f"{t:<12.1f} {precision:>10.3f} {recall:>10.3f} {f1:>10.3f} {int(fn):>10} {int(fp):>10}")

print("-" * 62)
print(f"\nBaseline comparison (Random Forest @ 0.3):")
print(f"  Recall: 0.831 | Precision: 0.812 | F1: 0.821 | FN: ~22")

Running threshold tuning on validation set...



Threshold     Precision     Recall         F1   FN count   FP count
--------------------------------------------------------------
0.5               0.000      0.000      0.000          0          0
0.4               0.000      0.000      0.000          0          0
0.3               0.000      0.000      0.000          0          0
0.2               0.000      0.000      0.000          0          0
0.1               0.000      0.000      0.000          0          0
--------------------------------------------------------------

Baseline comparison (Random Forest @ 0.3):
  Recall: 0.831 | Precision: 0.812 | F1: 0.821 | FN: ~22


In [14]:
print("Running threshold tuning on validation set...")

val_outputs = trainer.predict(val_dataset)

# --- Diagnose the raw outputs first ---
print("val_outputs.predictions shape:", val_outputs.predictions.shape)
print("val_outputs.predictions dtype:", val_outputs.predictions.dtype)
print("val_outputs.label_ids shape:", val_outputs.label_ids.shape)
print("val_outputs.label_ids dtype:", val_outputs.label_ids.dtype)
print("\nFirst 5 raw logits:")
print(val_outputs.predictions[:5])
print("\nFirst 5 labels from predict output:")
print(val_outputs.label_ids[:5])

# Use label_ids directly from trainer.predict() output
# rather than the separately-defined val_labels list
true_labels = val_outputs.label_ids

# Convert logits to probabilities
val_logits = val_outputs.predictions
val_probs = torch.softmax(torch.tensor(val_logits, dtype=torch.float32), dim=1).numpy()
val_probs_fraud = val_probs[:, 1]

print("\nSoftmax prob range: min={:.4f}, max={:.4f}".format(
    val_probs_fraud.min(), val_probs_fraud.max()))
print("Predictions at 0.5 (first 10):", (val_probs_fraud >= 0.5).astype(int)[:10])
print("True labels (first 10):", true_labels[:10])
print("Fraud count in true_labels:", (true_labels == 1).sum())
print("Fraud count in predictions at 0.5:", (val_probs_fraud >= 0.5).sum())

thresholds = [0.5, 0.4, 0.3, 0.2, 0.1]
threshold_results = []

print(f"\n{'Threshold':<12} {'Precision':>10} {'Recall':>10} {'F1':>10} {'FN count':>10} {'FP count':>10}")
print("-" * 62)

for t in thresholds:
    preds = (val_probs_fraud >= t).astype(int)

    tp = int(((preds == 1) & (true_labels == 1)).sum())
    fp = int(((preds == 1) & (true_labels == 0)).sum())
    fn = int(((preds == 0) & (true_labels == 1)).sum())

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    threshold_results.append({
        "threshold": t,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "fn_count": fn,
        "fp_count": fp,
    })

    print(f"{t:<12.1f} {precision:>10.3f} {recall:>10.3f} {f1:>10.3f} {fn:>10} {fp:>10}")

print("-" * 62)
print(f"\nBaseline comparison (Random Forest @ 0.3):")
print(f"  Recall: 0.831 | Precision: 0.812 | F1: 0.821 | FN: ~22")

Running threshold tuning on validation set...


val_outputs.predictions shape: (2682, 2)
val_outputs.predictions dtype: float32
val_outputs.label_ids shape: (2682,)
val_outputs.label_ids dtype: int64

First 5 raw logits:
[[ 4.4257812 -4.34375  ]
 [ 4.1523438 -4.2695312]
 [ 3.0625    -1.9208984]
 [ 4.421875  -4.2734375]
 [ 4.3320312 -4.265625 ]]

First 5 labels from predict output:
[0 0 0 0 0]

Softmax prob range: min=0.0001, max=0.9999
Predictions at 0.5 (first 10): [0 0 0 0 0 0 0 0 0 0]
True labels (first 10): [0 0 0 0 0 0 0 0 0 0]
Fraud count in true_labels: 130
Fraud count in predictions at 0.5: 124

Threshold     Precision     Recall         F1   FN count   FP count
--------------------------------------------------------------
0.5               0.879      0.838      0.858         21         15
0.4               0.866      0.846      0.856         20         17
0.3               0.846      0.846      0.846         20         20
0.2               0.846      0.846      0.846         20         20
0.1               0.827      0.846

In [15]:
print("Running final evaluation on held-out test set...")
print("Using decision threshold: 0.5 (selected on validation set)")
print("="*60)

test_outputs = trainer.predict(test_dataset)
test_logits = test_outputs.predictions
true_test_labels = test_outputs.label_ids

test_probs = torch.softmax(
    torch.tensor(test_logits, dtype=torch.float32), dim=1
).numpy()
test_probs_fraud = test_probs[:, 1]

FINAL_THRESHOLD = 0.5
test_preds = (test_probs_fraud >= FINAL_THRESHOLD).astype(int)

print("\nClassification Report:")
print(classification_report(
    true_test_labels, test_preds,
    target_names=["genuine", "fraudulent"]
))

print("ROC-AUC:", round(roc_auc_score(true_test_labels, test_probs_fraud), 4))

cm = confusion_matrix(true_test_labels, test_preds)
print("\nConfusion Matrix (rows=actual, cols=predicted):")
print("               Predicted Genuine   Predicted Fraudulent")
print(f"Actual Genuine       {cm[0][0]:>6}               {cm[0][1]:>6}")
print(f"Actual Fraudulent    {cm[1][0]:>6}               {cm[1][1]:>6}")

tp = cm[1][1]
fn = cm[1][0]
fp = cm[0][1]
tn = cm[0][0]
print(f"\nTrue Positives  (fraud caught):   {tp}")
print(f"False Negatives (fraud missed):   {fn}  ← minimize this")
print(f"False Positives (genuine flagged):{fp}")
print(f"True Negatives  (genuine correct):{tn}")

print("\n" + "="*60)
print("FINAL MODEL COMPARISON (Test Set)")
print("="*60)
print(f"{'Model':<30} {'Recall':>8} {'Precision':>10} {'F1':>8} {'ROC-AUC':>9}")
print("-"*60)
print(f"{'Random Forest @ 0.3':<30} {'0.831':>8} {'0.812':>10} {'0.821':>8} {'0.978':>9}")
report_dict = classification_report(
    true_test_labels, test_preds,
    target_names=["genuine", "fraudulent"],
    output_dict=True
)
bert_recall    = round(report_dict["fraudulent"]["recall"], 3)
bert_precision = round(report_dict["fraudulent"]["precision"], 3)
bert_f1        = round(report_dict["fraudulent"]["f1-score"], 3)
bert_auc       = round(roc_auc_score(true_test_labels, test_probs_fraud), 3)
print(f"{'BERT @ 0.5 (final)':<30} {bert_recall:>8} {bert_precision:>10} {bert_f1:>8} {bert_auc:>9}")
print("="*60)

Running final evaluation on held-out test set...
Using decision threshold: 0.5 (selected on validation set)



Classification Report:
              precision    recall  f1-score   support

     genuine       0.99      0.99      0.99      2552
  fraudulent       0.90      0.85      0.87       130

    accuracy                           0.99      2682
   macro avg       0.94      0.92      0.93      2682
weighted avg       0.99      0.99      0.99      2682

ROC-AUC: 0.9938

Confusion Matrix (rows=actual, cols=predicted):
               Predicted Genuine   Predicted Fraudulent
Actual Genuine         2539                   13
Actual Fraudulent        19                  111

True Positives  (fraud caught):   111
False Negatives (fraud missed):   19  ← minimize this
False Positives (genuine flagged):13
True Negatives  (genuine correct):2539

FINAL MODEL COMPARISON (Test Set)
Model                            Recall  Precision       F1   ROC-AUC
------------------------------------------------------------
Random Forest @ 0.3               0.831      0.812    0.821     0.978
BERT @ 0.5 (final)       

In [18]:
metadata_path = os.path.join(MODEL_DIR, "bert_metadata.json")

with open(metadata_path, "r") as f:
    metadata = json.load(f)

metadata["decision_threshold"] = 0.5
metadata["threshold_tuning_finding"] = (
    "Threshold tuning showed minimal recall improvement below 0.5 "
    "(FN reduced by only 1 posting across all lower thresholds). "
    "Default threshold 0.5 retained as optimal operating point — "
    "indicates well-calibrated model probabilities."
)
metadata["test_set_metrics"] = {
    "accuracy": round(report_dict["accuracy"], 4),
    "precision_fraud": bert_precision,
    "recall_fraud": bert_recall,
    "f1_fraud": bert_f1,
    "roc_auc": bert_auc,
    "true_positives": int(tp),
    "false_negatives": int(fn),
    "false_positives": int(fp),
    "true_negatives": int(tn),
    "total_test_samples": 2682,
    "total_fraudulent_in_test": 130,
}
metadata["comparison_vs_baseline"]["bert_test_recall"] = bert_recall
metadata["comparison_vs_baseline"]["bert_test_precision"] = bert_precision
metadata["comparison_vs_baseline"]["bert_test_f1"] = bert_f1
metadata["comparison_vs_baseline"]["bert_test_roc_auc"] = bert_auc
metadata["comparison_vs_baseline"]["bert_test_fn_count"] = int(fn)

with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2)

print("Updated bert_metadata.json with final test-set results.")
print("\nFinal metadata summary:")
print(f"  Decision threshold: {metadata['decision_threshold']}")
print(f"  Test recall:        {metadata['test_set_metrics']['recall_fraud']}")
print(f"  Test precision:     {metadata['test_set_metrics']['precision_fraud']}")
print(f"  Test F1:            {metadata['test_set_metrics']['f1_fraud']}")
print(f"  Test ROC-AUC:       {metadata['test_set_metrics']['roc_auc']}")
print(f"  False negatives:    {metadata['test_set_metrics']['false_negatives']}")

Updated bert_metadata.json with final test-set results.

Final metadata summary:
  Decision threshold: 0.5
  Test recall:        0.854
  Test precision:     0.895
  Test F1:            0.874
  Test ROC-AUC:       0.994
  False negatives:    19
